<a href="https://colab.research.google.com/github/Rye-create/Assessment-2/blob/main/Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Restore environment after a fresh Colab session ---
!pip install -q rank_bm25 arxiv pymupdf requests

from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/data
!cp -r /content/drive/MyDrive/cp5403/data/. /content/data/
!ls -lh /content/data

Mounted at /content/drive
total 748K
-rw------- 1 root root 219K Oct 26 09:39 epmc_metadata.csv
-rw------- 1 root root  28K Oct 26 09:39 extraction_papers.csv
drwx------ 2 root root 4.0K Oct 26 09:39 index
-rw------- 1 root root 374K Oct 26 09:39 papers_metadata.csv
drwx------ 2 root root  20K Oct 26 09:39 processed
-rw------- 1 root root  42K Oct 26 09:39 rag_papers.csv
drwx------ 3 root root 4.0K Oct 26 09:39 raw
-rw------- 1 root root  51K Oct 26 09:39 selected_papers.csv


In [2]:
!pip install arxiv pymupdf

In [3]:
import arxiv

client = arxiv.Client()
search = arxiv.Search(
    query='all:"laser powder bed fusion"',
    max_results=200,
    sort_by=arxiv.SortCriterion.Relevance,
)

# Only fetch metadata here; no PDFs are downloaded
results = list(client.results(search))
print("Papers found:", len(results))

# Print the first 20 titles for a quick manual check
for r in results[:20]:
    print(r.get_short_id(), r.title)

Papers found: 180
2011.09065v2 Towards Online Monitoring and Data-driven Control: A Study of Segmentation Algorithms for Laser Powder Bed Fusion Processes
2509.03255v1 Anti-scatter grid prototype manufactured via laser powder bed fusion of pure tungsten
2411.18048v1 MeltpoolINR: Predicting temperature field, melt pool geometry, and their rate of change in laser powder bed fusion
2201.09978v1 An empirical model for feedforward control of laser powder bed fusion
2102.03355v1 Thermal Control of Laser Powder Bed Fusion Using Deep Reinforcement Learning
2311.10218v2 Layer-to-Layer Melt Pool Control in Laser Powder Bed Fusion
1902.06785v1 Uncertainty Analysis of Microsegregation during Laser Powder Bed Fusion
2509.16114v1 Real-Time Thermal State Estimation and Forecasting in Laser Powder Bed Fusion
2509.01754v1 TransMatch: A Transfer-Learning Framework for Defect Detection in Laser Powder Bed Fusion Additive Manufacturing
2604.04208v1 Towards Agentic Defect Reasoning: A Graph-Assisted Retrie

In [4]:
import arxiv, pymupdf, requests, os, time

# Remember to add data/raw/ to .gitignore so PDFs are not pushed to GitHub
os.makedirs("data/raw", exist_ok=True)

client = arxiv.Client()
search = arxiv.Search(
    query='all:"laser powder bed fusion" AND all:"AlSi10Mg"',
    max_results=5,
    sort_by=arxiv.SortCriterion.Relevance,
)

for r in client.results(search):
    fname = r.get_short_id().replace("/", "_") + ".pdf"
    path = os.path.join("data/raw", fname)
    try:
        resp = requests.get(r.pdf_url, timeout=60)
        resp.raise_for_status()
        with open(path, "wb") as f:
            f.write(resp.content)
        print(r.title, "->", fname)
    except Exception as e:
        print("Download failed:", r.title, e)
    time.sleep(3)  # Be polite to arXiv: avoid rapid repeated requests

On the influence of powder particle size on single-track formation in laser powder bed fusion of AlSi10Mg alloy -> 2507.23422v1.pdf
Extension of a phase-field KKS model to predict the microstructure evolution in LPBF AlSi10Mg alloy submitted to non isothermal processes -> 2407.11580v1.pdf
Solidification Characteristics of Laser-Powder Bed Fused AlSi10Mg: Role of Building Direction -> 2003.09757v1.pdf
Laser Remelting for Reduced Porosity on Additively Manufactured Aluminium Mirrors -> 2607.21253v1.pdf
Additively manufactured ultra-high vacuum chamber below $10^{-10}$ mbar -> 1903.07708v3.pdf


In [5]:
import requests
from collections import Counter
from urllib.parse import urlparse

BASE = "https://api.openalex.org/works"
params = {
    "filter": ",".join([
        'title_and_abstract.search:"laser powder bed fusion"',
        "open_access.is_oa:true",
        "best_oa_location.license:cc-by",
        "type:article",
        "from_publication_date:2018-01-01",
    ]),
    "per-page": 200,
    "select": "id,doi,title,publication_year,best_oa_location",
    "mailto": "your_email@example.com",  # Replace with your own email
}

resp = requests.get(BASE, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Total matching works:", data["meta"]["count"])
works = data["results"]

def pdf_host(work):
    """Return the website host of the PDF link, or None if missing."""
    loc = work.get("best_oa_location") or {}
    url = loc.get("pdf_url")
    return urlparse(url).netloc if url else None

# Which websites do the PDF links come from?
host_counts = Counter(pdf_host(w) for w in works)
print("Top PDF hosts:", host_counts.most_common(10))

# Show non-review titles with their PDF host (first 25)
shown = 0
for w in works:
    title = w["title"] or ""
    if "review" in title.lower():
        continue
    print(w["publication_year"], "|", title, "|", pdf_host(w))
    shown += 1
    if shown >= 25:
        break

Total matching works: 4266
Top PDF hosts: [('www.mdpi.com', 54), (None, 50), ('link.springer.com', 31), ('www.sciencedirect.com', 29), ('www.nature.com', 8), ('ars.els-cdn.com', 5), ('onlinelibrary.wiley.com', 5), ('www.tandfonline.com', 3), ('www.frontiersin.org', 3), ('iopscience.iop.org', 2)]
2020 | Processing parameters in laser powder bed fusion metal additive manufacturing | www.sciencedirect.com
2018 | Melt pool temperature and cooling rates in laser powder bed fusion | ars.els-cdn.com
2020 | The role of side-branching in microstructure development in laser powder-bed fusion | www.nature.com
2018 | Laser powder bed fusion in high-pressure atmospheres | link.springer.com
2019 | Dynamics of pore formation during laser powder bed fusion additive manufacturing | www.nature.com
2018 | Laser powder bed fusion at sub-atmospheric pressures | www.sciencedirect.com
2019 | Benchmarking of Laser Powder Bed Fusion Machines | None
2019 | Additive manufacturing of glass with laser powder bed f

In [6]:
import requests
from collections import Counter

URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
params = {
    "query": '"laser powder bed fusion" AND OPEN_ACCESS:Y AND HAS_FT:Y',
    "format": "json",
    "resultType": "core",
    "pageSize": 100,
}

resp = requests.get(URL, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Total matching records:", data["hitCount"])

results = data["resultList"]["result"]

# License distribution on the first page
license_counts = Counter(r.get("license") for r in results)
print("License counts (first page):", license_counts)

# Print a few examples for a manual check
for r in results[:15]:
    print(r.get("pubYear"), "|", r.get("pmcid"), "|", r.get("license"),
          "|", (r.get("title") or "")[:80])

Total matching records: 1770
License counts (first page): Counter({'cc by': 83, 'cc by-nc-nd': 16, 'cc by-nc': 1})
2026 | PMC13608174 | cc by | Volumetric Infiltration of Brushite and Monetite Cements into Laser Powder Bed F
2026 | PMC13514805 | cc by | Investigation on the Microstructure and Compressive Properties of W-15Mo Alloy F
2026 | PMC13177226 | cc by-nc-nd | Correction to "Structural Design and Mechanical Properties of Metal Vascular Ste
2026 | PMC13608489 | cc by | Thermal-History-Mediated Defect-Microstructure-Property Evolution in Multi-Laser
2026 | PMC13514143 | cc by | Effect of Cu Particle Cross-Contamination in AlSi10Mg Powder Feedstock: Tensile 
2026 | PMC13691206 | cc by-nc | Cavitation erosion-corrosion properties of as-cast NiTi and LPBF NiTi in 3.5 wt%
2026 | PMC13467871 | cc by | Fitting-Free Diagnosis of Conduction-Model Breakdown in Laser Powder Bed Fusion.
2026 | PMC13514662 | cc by | Root-Inspired Bio-Interlocking Structure Design and Its Mechanism on Enhancin

In [8]:
import requests, csv, os, time

URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
params = {
    "query": '"laser powder bed fusion" AND OPEN_ACCESS:Y AND HAS_FT:Y',
    "format": "json",
    "resultType": "core",
    "pageSize": 1000,
    "cursorMark": "*",   # Cursor paging walks through all results
}

records = []
while True:
    resp = requests.get(URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    results = data["resultList"]["result"]
    if not results:
        break
    for r in results:
        records.append({
            "pmcid": r.get("pmcid"),
            "doi": r.get("doi"),
            "title": r.get("title"),
            "year": r.get("pubYear"),
            "journal": r.get("journalTitle"),
            "license": r.get("license"),
        })
    next_cursor = data.get("nextCursorMark")
    if not next_cursor or next_cursor == params["cursorMark"]:
        break
    params["cursorMark"] = next_cursor
    time.sleep(0.5)  # Be polite to the API

print("Total records fetched:", len(records))

# Keep only cc by licensed records with a PMCID
EXCLUDE_WORDS = ["correction", "erratum", "editorial", "review",
                 "retraction", "special issue"]

def keep(r):
    title = (r["title"] or "").lower()
    return (
        r["pmcid"]
        and r["license"] == "cc by"
        and not any(w in title for w in EXCLUDE_WORDS)
    )

filtered = [r for r in records if keep(r)]
print("After filtering (cc by, no corrections/editorials/reviews):", len(filtered))

os.makedirs("data", exist_ok=True)
with open("data/epmc_metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(filtered[0].keys()))
    writer.writeheader()
    writer.writerows(filtered)
print("Saved data/epmc_metadata.csv")

Total records fetched: 1770
After filtering (cc by, no corrections/editorials/reviews): 1403
Saved data/epmc_metadata.csv


In [9]:
import requests, csv, os, time

URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
params = {
    "query": '"laser powder bed fusion" AND OPEN_ACCESS:Y AND HAS_FT:Y',
    "format": "json",
    "resultType": "core",
    "pageSize": 100,      # Smaller pages are lighter for the server
    "cursorMark": "*",    # Cursor paging walks through all results
}

def get_with_retry(url, params, max_retries=5):
    """GET request that retries on temporary server errors."""
    wait = 3
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=60)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Temporary error {resp.status_code}")
            resp.raise_for_status()
            return resp
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout) as e:
            print(f"Attempt {attempt}/{max_retries} failed: {e}. Waiting {wait}s...")
            time.sleep(wait)
            wait *= 2
    raise RuntimeError("All retries failed. Try again later.")

records = []
page = 0
while True:
    resp = get_with_retry(URL, params)
    data = resp.json()
    results = data["resultList"]["result"]
    if not results:
        break
    for r in results:
        records.append({
            "pmcid": r.get("pmcid"),
            "doi": r.get("doi"),
            "title": r.get("title"),
            "year": r.get("pubYear"),
            "journal": r.get("journalTitle"),
            "license": r.get("license"),
        })
    page += 1
    print(f"Page {page} done, records so far: {len(records)}")
    next_cursor = data.get("nextCursorMark")
    if not next_cursor or next_cursor == params["cursorMark"]:
        break
    params["cursorMark"] = next_cursor
    time.sleep(1)  # Be polite to the API

print("Total records fetched:", len(records))

# Keep only "cc by" licensed records that have a PMCID
EXCLUDE_WORDS = ["correction", "erratum", "editorial", "review",
                 "retraction", "special issue"]

def keep(r):
    title = (r["title"] or "").lower()
    return (
        r["pmcid"]
        and r["license"] == "cc by"
        and not any(w in title for w in EXCLUDE_WORDS)
    )

filtered = [r for r in records if keep(r)]
print("After filtering (cc by, no corrections/editorials/reviews):", len(filtered))

os.makedirs("data", exist_ok=True)
with open("data/epmc_metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(filtered[0].keys()))
    writer.writeheader()
    writer.writerows(filtered)
print("Saved data/epmc_metadata.csv")

Page 1 done, records so far: 100
Page 2 done, records so far: 200
Page 3 done, records so far: 300
Page 4 done, records so far: 400
Page 5 done, records so far: 500
Page 6 done, records so far: 600
Page 7 done, records so far: 700
Page 8 done, records so far: 800
Page 9 done, records so far: 900
Page 10 done, records so far: 1000
Page 11 done, records so far: 1100
Page 12 done, records so far: 1200
Page 13 done, records so far: 1300
Page 14 done, records so far: 1400
Page 15 done, records so far: 1500
Page 16 done, records so far: 1600
Page 17 done, records so far: 1700
Page 18 done, records so far: 1770
Total records fetched: 1770
After filtering (cc by, no corrections/editorials/reviews): 1403
Saved data/epmc_metadata.csv


In [10]:
import csv, os, time, requests
import xml.etree.ElementTree as ET

os.makedirs("data/raw/xml", exist_ok=True)

with open("data/epmc_metadata.csv", encoding="utf-8") as f:
    papers = list(csv.DictReader(f))

BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest/{}/fullTextXML"
MAX_PAPERS = 20  # Small test first

for p in papers[:MAX_PAPERS]:
    path = os.path.join("data/raw/xml", p["pmcid"] + ".xml")
    if os.path.exists(path):
        continue
    try:
        resp = requests.get(BASE.format(p["pmcid"]), timeout=60)
        resp.raise_for_status()
        with open(path, "wb") as out:
            out.write(resp.content)
    except Exception as e:
        print("FAILED:", p["pmcid"], e)
    time.sleep(1)

# Inspect structure: section titles and number of tables per paper
for fname in sorted(os.listdir("data/raw/xml"))[:5]:
    root = ET.parse(os.path.join("data/raw/xml", fname)).getroot()
    sec_titles = []
    for sec in root.iter("sec"):
        t = sec.find("title")
        if t is not None:
            sec_titles.append("".join(t.itertext()).strip())
    n_tables = len(list(root.iter("table-wrap")))
    print(fname, "| tables:", n_tables)
    print("   sections:", sec_titles[:12])

PMC10055713.xml | tables: 4
   sections: ['1. Introduction', '2. Materials and Methods', '2.1. Approach', '2.2. Geometry', '2.3. Boundary Conditions', '2.4. Layer-by-Layer Manufacturing', '2.5. Material', '2.6. Heat Source', '2.7. Scanning Strategy', '2.8. Parameter Study—Modeled Manufacturing Parameters', '2.9. Simplifications', '3. Results']
PMC10056913.xml | tables: 4
   sections: ['1. Introduction', '2. Materials and Methods', '2.1. Materials', '2.2. LPBF Process', '2.3. Characterizations', '3. Results and Discussion', '3.1. Influence of Laser Parameters on the Relative Density', '3.2. Influence of Laser Parameters on the Surface Roughness', '3.3. Microstructure', '3.4. Compression Properties', '4. Conclusions']
PMC10095620.xml | tables: 2
   sections: ['1. Introduction', '2. Materials and Methods', '3. Results and Discussion', '3.1. Powder Mixture Characterization', '3.2. Characterization of the As-Built Sample', '3.3. Characterization of the Heat-Treated Sample', '3.4. Microhardn

In [11]:
import os, json
import xml.etree.ElementTree as ET

def clean(el):
    """Return the text of an element with whitespace normalized."""
    return " ".join("".join(el.itertext()).split())

def walk(sec, path, out):
    """Recursively collect paragraphs with their section path."""
    t = sec.find("title")
    title = clean(t) if t is not None else ""
    new_path = path + [title] if title else path
    for child in sec:
        if child.tag == "p":
            out.append({"section": " > ".join(new_path), "text": clean(child)})
        elif child.tag == "sec":
            walk(child, new_path, out)

def parse_paper(xml_path):
    root = ET.parse(xml_path).getroot()

    title_el = root.find(".//article-title")
    title = clean(title_el) if title_el is not None else ""

    paragraphs = []
    body = root.find(".//body")
    if body is not None:
        for child in body:
            if child.tag == "sec":
                walk(child, [], paragraphs)
            elif child.tag == "p":
                paragraphs.append({"section": "", "text": clean(child)})

    tables = []
    for tw in root.iter("table-wrap"):
        label = tw.find("label")
        caption = tw.find("caption")
        rows = []
        for tr in tw.iter("tr"):
            cells = [clean(c) for c in tr if c.tag in ("th", "td")]
            if cells:
                rows.append(cells)
        tables.append({
            "label": clean(label) if label is not None else "",
            "caption": clean(caption) if caption is not None else "",
            "rows": rows,
        })

    return {"title": title, "paragraphs": paragraphs, "tables": tables}

KEYWORDS = ["laser power", "scan speed", "scanning speed", "scan velocity",
            "scanning velocity", "hatch", "layer thickness"]

os.makedirs("data/processed", exist_ok=True)

for fname in sorted(os.listdir("data/raw/xml")):
    pmcid = fname.replace(".xml", "")
    paper = parse_paper(os.path.join("data/raw/xml", fname))

    with open(f"data/processed/{pmcid}.json", "w", encoding="utf-8") as f:
        json.dump(paper, f, ensure_ascii=False, indent=2)

    has_methods = any(
        ("method" in p["section"].lower() or "experiment" in p["section"].lower())
        for p in paper["paragraphs"]
    )
    table_text = " ".join(
        " ".join(" ".join(r) for r in t["rows"]) + " " + t["caption"]
        for t in paper["tables"]
    )
    all_text = (" ".join(p["text"] for p in paper["paragraphs"]) + " " + table_text).lower()
    kw_hits = sum(all_text.count(k) for k in KEYWORDS)

    print(pmcid, "| paragraphs:", len(paper["paragraphs"]),
          "| tables:", len(paper["tables"]),
          "| has_methods:", has_methods,
          "| keyword hits:", kw_hits)
    print("   title:", paper["title"][:90])

# Print the first table of one paper to see what a parameter table looks like
sample = json.load(open("data/processed/PMC12985775.json", encoding="utf-8"))
if sample["tables"]:
    t = sample["tables"][0]
    print("\nSample table:", t["label"], "-", t["caption"][:120])
    for row in t["rows"][:8]:
        print("  ", row)

PMC10055713 | paragraphs: 41 | tables: 4 | has_methods: True | keyword hits: 40
   title: Residual Stress Formation Mechanisms in Laser Powder Bed Fusion—A Numerical Evaluation
PMC10056913 | paragraphs: 24 | tables: 4 | has_methods: True | keyword hits: 129
   title: Process Optimization and Tailored Mechanical Properties of a Nuclear Zr-4 Alloy Fabricated
PMC10095620 | paragraphs: 29 | tables: 2 | has_methods: True | keyword hits: 3
   title: Fabrication of Ti3Al-Based Intermetallic Alloy by Laser Powder Bed Fusion Using a Powder M
PMC10095776 | paragraphs: 30 | tables: 1 | has_methods: True | keyword hits: 2
   title: Powder Spreading Mechanism in Laser Powder Bed Fusion Additive Manufacturing: Experiments 
PMC10131821 | paragraphs: 24 | tables: 0 | has_methods: True | keyword hits: 10
   title: Metallic 4D Printing of Laser Stimulation
PMC10143746 | paragraphs: 23 | tables: 2 | has_methods: True | keyword hits: 13
   title: Nanoscale Clustering in an Additively Manufactured Zr-Based

In [12]:
import json

paper = json.load(open("data/processed/PMC13164529.json", encoding="utf-8"))
print("TITLE:", paper["title"])

# Print Methods/Experimental paragraphs that mention key parameters
KEYWORDS = ["laser power", "scan speed", "scanning speed", "hatch", "layer thickness"]

print("\n=== Methods paragraphs with parameter keywords ===")
for p in paper["paragraphs"]:
    sec = p["section"].lower()
    if ("method" in sec or "experiment" in sec) and any(k in p["text"].lower() for k in KEYWORDS):
        print("[", p["section"], "]")
        print(p["text"][:600])
        print("---")

# Print the tables with captions and the first rows
print("\n=== Tables ===")
for t in paper["tables"]:
    print(t["label"], "-", t["caption"][:150])
    for row in t["rows"][:6]:
        print("  ", row)
    print()

TITLE: Effects of Scan Speed on Crack Elimination, Microstructural Evolution, and Mechanical Properties of IN738LC Alloy Processed by Laser Powder Bed Fusion

=== Methods paragraphs with parameter keywords ===
[ 2. Experiment and Simulation > 2.2. LPBF Process ]
The LPBF forming equipment (DiMetal-280, South China University of Technology, Guangzhou, China) is fitted with a 500 W IPG-YLR-500 continuous fiber laser, featuring the beam diameter of 90 μm at the focal plane. To prevent sample oxidation, the entire LPBF process is operated in an argon atmosphere (pressure below 0.08 bar), and an oxygen analyzer is used to maintain the oxygen content within the chamber below 100 ppm. To ensure excellent material bonding, the samples are fabricated on 316L stainless steel substrates at room-temperature. Figure 2a illustrates the orthogonal scan strategy impl
---
[ 2. Experiment and Simulation > 2.2. LPBF Process ]
As shown in Table 2, the range of calculated VED values is 48.6 to 114.6 J/mm3.

In [13]:
import csv, os, time, requests

os.makedirs("data/raw/xml", exist_ok=True)

with open("data/epmc_metadata.csv", encoding="utf-8") as f:
    papers = list(csv.DictReader(f))

BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest/{}/fullTextXML"
TARGET = 400  # Number of papers to have in data/raw/xml

def download(pmcid, retries=3):
    wait = 3
    for _ in range(retries):
        try:
            resp = requests.get(BASE.format(pmcid), timeout=60)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Temporary error {resp.status_code}")
            resp.raise_for_status()
            return resp.content
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout):
            time.sleep(wait)
            wait *= 2
    return None

have = len(os.listdir("data/raw/xml"))
failed = 0
for p in papers:
    if have >= TARGET:
        break
    path = os.path.join("data/raw/xml", p["pmcid"] + ".xml")
    if os.path.exists(path):
        continue
    content = download(p["pmcid"])
    if content is None:
        failed += 1
        print("FAILED:", p["pmcid"])
        continue
    with open(path, "wb") as out:
        out.write(content)
    have += 1
    if have % 25 == 0:
        print("Downloaded so far:", have)
    time.sleep(1)  # Be polite to the API

print("Done. XML files:", have, "| Failed:", failed)

Done. XML files: 400 | Failed: 0


In [14]:
import os, json, csv
from collections import Counter
import xml.etree.ElementTree as ET

def clean(el):
    """Return the text of an element with whitespace normalized."""
    return " ".join("".join(el.itertext()).split())

def walk(sec, path, out):
    """Recursively collect paragraphs with their section path."""
    t = sec.find("title")
    title = clean(t) if t is not None else ""
    new_path = path + [title] if title else path
    for child in sec:
        if child.tag == "p":
            out.append({"section": " > ".join(new_path), "text": clean(child)})
        elif child.tag == "sec":
            walk(child, new_path, out)

def parse_paper(xml_path):
    root = ET.parse(xml_path).getroot()
    title_el = root.find(".//article-title")
    title = clean(title_el) if title_el is not None else ""

    paragraphs = []
    body = root.find(".//body")
    if body is not None:
        for child in body:
            if child.tag == "sec":
                walk(child, [], paragraphs)
            elif child.tag == "p":
                paragraphs.append({"section": "", "text": clean(child)})

    tables = []
    for tw in root.iter("table-wrap"):
        label = tw.find("label")
        caption = tw.find("caption")
        rows = []
        for tr in tw.iter("tr"):
            cells = [clean(c) for c in tr if c.tag in ("th", "td")]
            if cells:
                rows.append(cells)
        tables.append({
            "label": clean(label) if label is not None else "",
            "caption": clean(caption) if caption is not None else "",
            "rows": rows,
        })
    return {"title": title, "paragraphs": paragraphs, "tables": tables}

KEYWORDS = ["laser power", "scan speed", "scanning speed", "scan velocity",
            "scanning velocity", "hatch", "layer thickness"]

# Load metadata so we can report year and journal for each paper
with open("data/epmc_metadata.csv", encoding="utf-8") as f:
    meta = {r["pmcid"]: r for r in csv.DictReader(f)}

os.makedirs("data/processed", exist_ok=True)
stats, errors = [], 0

for fname in sorted(os.listdir("data/raw/xml")):
    pmcid = fname.replace(".xml", "")
    try:
        paper = parse_paper(os.path.join("data/raw/xml", fname))
    except ET.ParseError:
        errors += 1
        print("Parse error:", pmcid)
        continue

    with open(f"data/processed/{pmcid}.json", "w", encoding="utf-8") as f:
        json.dump(paper, f, ensure_ascii=False)

    has_methods = any(
        ("method" in p["section"].lower() or "experiment" in p["section"].lower())
        for p in paper["paragraphs"]
    )
    table_text = " ".join(
        " ".join(" ".join(r) for r in t["rows"]) + " " + t["caption"]
        for t in paper["tables"]
    )
    all_text = (" ".join(p["text"] for p in paper["paragraphs"]) + " " + table_text).lower()
    kw_hits = sum(all_text.count(k) for k in KEYWORDS)

    stats.append({
        "pmcid": pmcid,
        "title": paper["title"],
        "year": meta.get(pmcid, {}).get("year"),
        "journal": meta.get(pmcid, {}).get("journal"),
        "paragraphs": len(paper["paragraphs"]),
        "tables": len(paper["tables"]),
        "has_methods": has_methods,
        "kw_hits": kw_hits,
    })

def keep(s):
    return (
        s["has_methods"]
        and s["paragraphs"] >= 20
        and (s["tables"] >= 1 or s["kw_hits"] >= 5)
    )

selected = [s for s in stats if keep(s)]
print("Parsed:", len(stats), "| Parse errors:", errors)
print("Selected after quality filter:", len(selected))
print("Selected by year:", sorted(Counter(s["year"] for s in selected).items()))
print("Top journals:", Counter(s["journal"] for s in selected).most_common(5))

with open("data/selected_papers.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(selected[0].keys()))
    writer.writeheader()
    writer.writerows(selected)
print("Saved data/selected_papers.csv")

Parsed: 400 | Parse errors: 0
Selected after quality filter: 321
Selected by year: [('2022', 4), ('2023', 39), ('2024', 74), ('2025', 102), ('2026', 101), ('2027', 1)]
Top journals: [('', 321)]
Saved data/selected_papers.csv


In [15]:
!pip install rank_bm25

In [16]:
# pip install rank_bm25  (run "!pip install rank_bm25" in a separate cell first)
import os, json, csv, re
from rank_bm25 import BM25Okapi

MAX_WORDS = 200

def make_chunks(pmcid, paper):
    """Split one parsed paper into section-aware text chunks."""
    chunks = []
    title = paper["title"]

    # Group consecutive paragraphs of the same section until MAX_WORDS
    buffer, buffer_words, current_section = [], 0, None

    def flush():
        nonlocal buffer, buffer_words
        if buffer:
            chunks.append({
                "pmcid": pmcid, "title": title, "section": current_section,
                "type": "text", "text": " ".join(buffer),
            })
        buffer, buffer_words = [], 0

    for p in paper["paragraphs"]:
        n_words = len(p["text"].split())
        if p["section"] != current_section or buffer_words + n_words > MAX_WORDS:
            flush()
            current_section = p["section"]
        buffer.append(p["text"])
        buffer_words += n_words
    flush()

    # Each table becomes its own chunk (caption + rows)
    for t in paper["tables"]:
        rows = [" | ".join(r) for r in t["rows"]]
        chunks.append({
            "pmcid": pmcid, "title": title, "section": "TABLE",
            "type": "table",
            "text": t["label"] + " " + t["caption"] + "\n" + "\n".join(rows),
        })
    return chunks

# Build chunks for the selected papers
with open("data/selected_papers.csv", encoding="utf-8") as f:
    selected = [r["pmcid"] for r in csv.DictReader(f)]

all_chunks = []
for pmcid in selected:
    paper = json.load(open(f"data/processed/{pmcid}.json", encoding="utf-8"))
    all_chunks.extend(make_chunks(pmcid, paper))

print("Papers:", len(selected), "| Chunks:", len(all_chunks))
lengths = [len(c["text"].split()) for c in all_chunks]
print("Average words per chunk:", sum(lengths) // len(lengths), "| Max:", max(lengths))

# Save chunks (small enough to keep locally; not for GitHub)
os.makedirs("data/index", exist_ok=True)
with open("data/index/chunks.jsonl", "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# BM25 baseline: lowercase words and numbers, keep units like "mm/s" split into tokens
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in all_chunks])

def search(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    top = scores.argsort()[::-1][:k]
    for rank, i in enumerate(top, 1):
        c = all_chunks[i]
        print(f"#{rank} score={scores[i]:.1f} | {c['pmcid']} | {c['section']}")
        print("   paper:", c["title"][:80])
        print("   text :", c["text"][:250].replace("\n", " "))
    print()

# Three test questions
search("What laser power and scan speed were used for IN738LC")
search("effect of base plate preheating on cracking in 7075 aluminum")
search("laser power layer thickness hatch spacing for copper alloy")

Papers: 321 | Chunks: 13091
Average words per chunk: 147 | Max: 2781
#1 score=24.9 | PMC10890519 | 1. Introduction
   paper: Understanding Melt Pool Behavior of 316L Stainless Steel in Laser Powder Bed Fus
   text : One of the direct impacts of the solidified melt tracks is the surface topography and roughness resulting from melt pool morphologies. In LPBF, the surface of a part forms through the layer-by-layer deposition of melted powder material, which melts a
#2 score=24.9 | PMC13164529 | 1. Introduction
   paper: Effects of Scan Speed on Crack Elimination, Microstructural Evolution, and Mecha
   text : It is notable that as one of the core parameters in LPBF, scan speed directly influences the thermal behavior of the melt pool, microstructure and corresponding mechanical properties [21]. Studies indicate that scan speed significantly affects the te
#3 score=24.5 | PMC13164529 | 2. Experiment and Simulation > 2.2. LPBF Process
   paper: Effects of Scan Speed on Crack Elimination, Mi

In [17]:
import re
from rank_bm25 import BM25Okapi

MAX_WORDS_ALLOWED = 300
WINDOW = 200

# 1) How many chunks are too long?
long_chunks = [c for c in all_chunks if len(c["text"].split()) > MAX_WORDS_ALLOWED]
print("Chunks over", MAX_WORDS_ALLOWED, "words:", len(long_chunks))
for c in sorted(long_chunks, key=lambda c: -len(c["text"].split()))[:5]:
    print("  ", len(c["text"].split()), "words |", c["type"], "|", c["section"][:50])

# 2) Split long chunks
def split_long(chunk):
    words = chunk["text"].split()
    if len(words) <= MAX_WORDS_ALLOWED:
        return [chunk]
    pieces = []
    if chunk["type"] == "table":
        # Split by rows and repeat the first line (label + caption) in every piece
        lines = chunk["text"].split("\n")
        header, rows = lines[0], lines[1:]
        buf, n = [], 0
        for row in rows:
            w = len(row.split())
            if buf and n + w > WINDOW:
                pieces.append(header + "\n" + "\n".join(buf))
                buf, n = [], 0
            buf.append(row)
            n += w
        if buf:
            pieces.append(header + "\n" + "\n".join(buf))
    else:
        for i in range(0, len(words), WINDOW):
            pieces.append(" ".join(words[i:i + WINDOW]))
    return [{**chunk, "text": t} for t in pieces]

chunks_v2 = []
for c in all_chunks:
    chunks_v2.extend(split_long(c))
print("Chunks before:", len(all_chunks), "| after splitting:", len(chunks_v2))

def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text.lower())

# 3) Two indexes: v1 = text only (previous baseline), v2 = title + section + text
def with_header(c):
    return f"{c['title']}. {c['section']}. {c['text']}"

bm25_v1 = BM25Okapi([tokenize(c["text"]) for c in chunks_v2])
bm25_v2 = BM25Okapi([tokenize(with_header(c)) for c in chunks_v2])

def search(index, query, k=3):
    scores = index.get_scores(tokenize(query))
    for rank, i in enumerate(scores.argsort()[::-1][:k], 1):
        c = chunks_v2[i]
        print(f"  #{rank} score={scores[i]:.1f} | {c['pmcid']} | {c['section'][:45]}")
        print("      paper:", c["title"][:75])

questions = [
    "What laser power and scan speed were used for IN738LC",
    "effect of base plate preheating on cracking in 7075 aluminum",
    "laser power layer thickness hatch spacing for copper alloy",
]
for q in questions:
    print("\nQUESTION:", q)
    print(" v1 (text only):")
    search(bm25_v1, q)
    print(" v2 (title + section + text):")
    search(bm25_v2, q)

Chunks over 300 words: 664
   2781 words | table | TABLE
   1968 words | text | Introduction
   1895 words | table | TABLE
   1107 words | text | Results and Discussion > Formation Mechanism of Ti
   1081 words | text | Results
Chunks before: 13091 | after splitting: 14070

QUESTION: What laser power and scan speed were used for IN738LC
 v1 (text only):
  #1 score=24.9 | PMC13164529 | 1. Introduction
      paper: Effects of Scan Speed on Crack Elimination, Microstructural Evolution, and 
  #2 score=24.8 | PMC10890519 | 1. Introduction
      paper: Understanding Melt Pool Behavior of 316L Stainless Steel in Laser Powder Be
  #3 score=24.4 | PMC13164529 | 2. Experiment and Simulation > 2.2. LPBF Proc
      paper: Effects of Scan Speed on Crack Elimination, Microstructural Evolution, and 
 v2 (title + section + text):
  #1 score=29.2 | PMC13164529 | 2. Experiment and Simulation > 2.2. LPBF Proc
      paper: Effects of Scan Speed on Crack Elimination, Microstructural Evolution, and 
  #2 s

In [18]:
# Starter evaluation set: question -> paper that should contain the answer
# NOTE: gold_pmcid values were derived from paper titles; verify each one by reading the paper.
eval_set = [
    {"q": "How does scan speed affect cracking in a nickel-based superalloy?",     "gold_pmcid": "PMC13164529"},
    {"q": "Does preheating the base plate reduce hot cracking in 7075 aluminium?", "gold_pmcid": "PMC12985958"},
    {"q": "How does heat treatment change the hardness of a CuNi2.5SiCr alloy?",   "gold_pmcid": "PMC12985775"},
    {"q": "How does copper contamination of AlSi10Mg powder affect tensile properties?", "gold_pmcid": "PMC13514143"},
    {"q": "How does residual stress change after aging heat treatment of CM247LC?","gold_pmcid": "PMC13630886"},
    {"q": "What are the compressive properties of a W-15Mo alloy?",                "gold_pmcid": "PMC13514805"},
    {"q": "What is the microstructure of 1080 plain carbon steel after printing?", "gold_pmcid": "PMC13027643"},
    {"q": "Which parameters of a block-shaped support tooth structure were optimized with an orthogonal experiment?", "gold_pmcid": "PMC13118256"},
]

def recall_at_k(index, k):
    """Fraction of questions whose gold paper appears among the top-k chunks."""
    hits = 0
    for item in eval_set:
        scores = index.get_scores(tokenize(item["q"]))
        top = scores.argsort()[::-1][:k]
        if any(chunks_v2[i]["pmcid"] == item["gold_pmcid"] for i in top):
            hits += 1
    return hits / len(eval_set)

for name, idx in [("v1 text only", bm25_v1), ("v2 title+section+text", bm25_v2)]:
    print(name, {f"Recall@{k}": round(recall_at_k(idx, k), 2) for k in (1, 3, 5, 10)})

v1 text only {'Recall@1': 0.62, 'Recall@3': 0.62, 'Recall@5': 0.62, 'Recall@10': 0.75}
v2 title+section+text {'Recall@1': 0.62, 'Recall@3': 0.62, 'Recall@5': 0.62, 'Recall@10': 0.62}


In [19]:
in_index = {c["pmcid"] for c in chunks_v2}

def diagnose(index, name):
    print("==", name)
    for item in eval_set:
        gold = item["gold_pmcid"]
        if gold not in in_index:
            print(f"gold {gold} NOT IN INDEX | Q: {item['q'][:60]}")
            continue
        scores = index.get_scores(tokenize(item["q"]))
        order = scores.argsort()[::-1]
        rank = next(r for r, i in enumerate(order, 1) if chunks_v2[i]["pmcid"] == gold)
        print(f"rank of first gold chunk: {rank} | Q: {item['q'][:60]}")
        if rank > 5:
            top = chunks_v2[order[0]]
            print("   top-1 paper:", top["pmcid"], "|", top["title"][:70])

diagnose(bm25_v1, "v1 text only")
diagnose(bm25_v2, "v2 title+section+text")

== v1 text only
rank of first gold chunk: 7 | Q: How does scan speed affect cracking in a nickel-based supera
   top-1 paper: PMC12250662 | Laser Powder Bed Fusion Additive Manufacturing of a CoCrFeNiCu High-En
rank of first gold chunk: 1 | Q: Does preheating the base plate reduce hot cracking in 7075 a
rank of first gold chunk: 1 | Q: How does heat treatment change the hardness of a CuNi2.5SiCr
gold PMC13514143 NOT IN INDEX | Q: How does copper contamination of AlSi10Mg powder affect tens
gold PMC13630886 NOT IN INDEX | Q: How does residual stress change after aging heat treatment o
rank of first gold chunk: 1 | Q: What are the compressive properties of a W-15Mo alloy?
rank of first gold chunk: 1 | Q: What is the microstructure of 1080 plain carbon steel after 
rank of first gold chunk: 1 | Q: Which parameters of a block-shaped support tooth structure w
== v2 title+section+text
rank of first gold chunk: 11 | Q: How does scan speed affect cracking in a nickel-based supera
   top-1 pape

In [20]:
import json, csv
from collections import Counter
from rank_bm25 import BM25Okapi

# Relaxed quality rule: needs a Methods section and enough body text
def keep_relaxed(s):
    return s["has_methods"] and s["paragraphs"] >= 20

selected_relaxed = [s for s in stats if keep_relaxed(s)]
print("Selected (relaxed rule):", len(selected_relaxed), "| previous rule: 321")

with open("data/selected_papers.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(selected_relaxed[0].keys()))
    writer.writeheader()
    writer.writerows(selected_relaxed)

# Rebuild chunks and both BM25 indexes
all_chunks = []
for s in selected_relaxed:
    paper = json.load(open(f"data/processed/{s['pmcid']}.json", encoding="utf-8"))
    all_chunks.extend(make_chunks(s["pmcid"], paper))

chunks_v2 = []
for c in all_chunks:
    chunks_v2.extend(split_long(c))
print("Chunks:", len(chunks_v2))

bm25_v1 = BM25Okapi([tokenize(c["text"]) for c in chunks_v2])
bm25_v2 = BM25Okapi([tokenize(with_header(c)) for c in chunks_v2])

# Check that every gold paper is now in the index, then re-score
in_index = {c["pmcid"] for c in chunks_v2}
print("Gold papers missing from index:",
      [i["gold_pmcid"] for i in eval_set if i["gold_pmcid"] not in in_index])

for name, idx in [("v1 text only", bm25_v1), ("v2 title+section+text", bm25_v2)]:
    print(name, {f"Recall@{k}": round(recall_at_k(idx, k), 2) for k in (1, 3, 5, 10)})

Selected (relaxed rule): 343 | previous rule: 321
Chunks: 14841
Gold papers missing from index: []
v1 text only {'Recall@1': 0.88, 'Recall@3': 0.88, 'Recall@5': 0.88, 'Recall@10': 1.0}
v2 title+section+text {'Recall@1': 0.88, 'Recall@3': 0.88, 'Recall@5': 0.88, 'Recall@10': 1.0}


In [21]:
import random, json

random.seed(42)  # Fixed seed so you get the same papers each time
sample_ids = random.sample([s["pmcid"] for s in selected_relaxed], 15)

for pmcid in sample_ids:
    paper = json.load(open(f"data/processed/{pmcid}.json", encoding="utf-8"))
    print("=" * 80)
    print(pmcid, "|", paper["title"])
    # Prefer paragraphs from the Conclusions section; otherwise the first paragraphs
    concl = [p["text"] for p in paper["paragraphs"] if "conclusion" in p["section"].lower()]
    texts = concl[:2] if concl else [p["text"] for p in paper["paragraphs"][:2]]
    for t in texts:
        print("  -", t[:500])

PMC13591421 | High-performance graphene/aluminum composites: progress and challenges from an industrial perspective
  - Graphene/aluminum composites exhibit exceptional mechanical, thermal, and electrical properties, and they are positioned as transformative materials for high-performance industrial applications. How to produce materials with high performance at a low cost is regarded as the most significant issue from the perspective of industrialization. The industrialization of high-performance Gr/Al composites is currently hindered by the C–Al interface, which is inherently unstable and difficult to regulate.
  - The most feasible solution is low-cost surface modification of graphene. Instead of directly mixing pristine graphene with aluminum, the graphene surface is coated or modified with a thin and uniform transition layer. This coating must protect the carbon material from erosion by aluminum melt and inhibit the formation of Al4C3. Besides, it must convert the interface from a

In [22]:
import json, re
from collections import Counter

LPBF_PATTERN = re.compile(
    r"laser powder bed fusion|powder bed fusion|\bLPBF\b|\bL-PBF\b|\bPBF-LB\b|"
    r"selective laser melting|\bSLM\b", re.IGNORECASE)

rows = []
for s in selected_relaxed:
    paper = json.load(open(f"data/processed/{s['pmcid']}.json", encoding="utf-8"))
    body = " ".join(p["text"] for p in paper["paragraphs"])
    n_body = len(LPBF_PATTERN.findall(body))
    in_title = bool(LPBF_PATTERN.search(paper["title"]))
    rows.append((s["pmcid"], n_body, in_title, paper["title"]))

counts = [r[1] for r in rows]
print("LPBF mentions in body, distribution:")
for lo, hi in [(0, 0), (1, 2), (3, 5), (6, 10), (11, 20), (21, 10**6)]:
    print(f"  {lo}-{hi if hi < 10**6 else '+'}: {sum(lo <= c <= hi for c in counts)}")

print("\nLowest 15 papers by mention count:")
for pmcid, n, in_title, title in sorted(rows, key=lambda r: r[1])[:15]:
    print(f"  {n:3d} | title_has_term={in_title} | {pmcid} | {title[:70]}")

LPBF mentions in body, distribution:
  0-0: 3
  1-2: 17
  3-5: 26
  6-10: 46
  11-20: 85
  21-+: 166

Lowest 15 papers by mention count:
    0 | title_has_term=False | PMC11434014 | The Role of Dislocation Type in the Thermal Stability of Cellular Stru
    0 | title_has_term=False | PMC13208258 | A Study on the Process, Microstructure, and Properties of Laser Additi
    0 | title_has_term=False | PMC13413770 | Comparative Study of Machine Learning Models for Optimal Prediction of
    1 | title_has_term=False | PMC13117853 | Microstructure Evolution and Solute Segregation of Inconel 718 in Lase
    1 | title_has_term=False | PMC13230510 | Evaluation of surface roughness of titanium implants on human fibrobla
    1 | title_has_term=False | PMC13566667 | Influence of Helium and Nitrogen as Quenching Atmospheres on the Amorp
    1 | title_has_term=False | PMC13609239 | In Situ Fabrication of Complex Hollow Nickel Microstructures via Filam
    2 | title_has_term=True | PMC10975002 | Develop

In [23]:
import random
random.seed(1)

border_low = [r for r in rows if 3 <= r[1] <= 5]    # 26 papers: print all
border_mid = [r for r in rows if 6 <= r[1] <= 10]   # 46 papers: print a sample of 15

print("=== 3-5 mentions (all) ===")
for pmcid, n, in_title, title in sorted(border_low, key=lambda r: r[1]):
    print(f"{n:2d} | {pmcid} | {title[:85]}")

print("\n=== 6-10 mentions (random sample of 15) ===")
for pmcid, n, in_title, title in random.sample(border_mid, 15):
    print(f"{n:2d} | {pmcid} | {title[:85]}")

=== 3-5 mentions (all) ===
 3 | PMC11252327 | High-pressure phase transition in 3-D printed nanolamellar high-entropy alloy by imag
 3 | PMC13258355 | Ultrasonic Evaluation of the Adhesion Between Regenerative Coatings and Steel Substra
 3 | PMC13389475 | Comprehensive study of the influence of reinforcements on the deformation mechanism a
 3 | PMC13467613 | Characterization of 2D PLA Structural Metamaterials—Methodology and Challenges for Pr
 3 | PMC13548926 | Geometry‐Encoded Programmable Mechanics in Single‐Material Dual‐Phase Metamaterials
 3 | PMC13566951 | AlSi10Mg Heat Sinks for Passive Cooling: Convective–Radiative Performance of Finned a
 3 | PMC13608489 | Thermal-History-Mediated Defect–Microstructure–Property Evolution in Multi-Laser Powd
 4 | PMC10859257 | Dataset of process-structure-property feature relationships for AlSi10Mg material fab
 4 | PMC11434085 | On the Creation of a Material Bond between L-PBF-Manufactured AZ91 and Ti-6Al-4V Comp
 4 | PMC12029075 | Fabrication

In [24]:
import random
random.seed(7)

kw = {s["pmcid"]: s["kw_hits"] for s in stats}
title_of = {r[0]: r[3] for r in rows}
mentions = {r[0]: r[1] for r in rows}

MIN_MENTIONS = 3
MIN_KW_HITS = 5

kept = [p for p in mentions if mentions[p] >= MIN_MENTIONS and kw[p] >= MIN_KW_HITS]
print("Kept by the two-signal rule:", len(kept), "of", len(mentions))

# Check 1: papers the old rule (>= 6 mentions) kept but the new rule drops
dropped_old_kept = [p for p in mentions if mentions[p] >= 6 and p not in kept]
print("\nDropped although mentions >= 6:", len(dropped_old_kept))
for p in random.sample(dropped_old_kept, min(12, len(dropped_old_kept))):
    print(f"  m={mentions[p]:3d} kw={kw[p]:3d} | {p} | {title_of[p][:80]}")

# Check 2: papers with only 3-5 mentions that the new rule keeps
rescued = [p for p in kept if mentions[p] <= 5]
print("\nRescued (3-5 mentions, kw >= 5):", len(rescued))
for p in rescued:
    print(f"  m={mentions[p]:3d} kw={kw[p]:3d} | {p} | {title_of[p][:80]}")

Kept by the two-signal rule: 202 of 343

Dropped although mentions >= 6: 106
  m=  7 kw=  4 | PMC11819817 | A Comparison Between the Residual Stresses of Ti6Al4V and Ti-6Al-2Sn-4Zr-6Mo Pro
  m= 10 kw=  0 | PMC11000381 | Additive manufacturing inert gas flow path strategies for multi-laser powder bed
  m= 35 kw=  4 | PMC12111142 | The Anisotropic Osteoinductive Capacity of a Nickel–Titanium Alloy Fabricated Th
  m=  9 kw=  2 | PMC13154604 | Multi-Stage micro-CT–based dimensional accuracy assessment of a voronoi-based pa
  m= 22 kw=  1 | PMC10608241 | Impact of Notches on Additively Manufactured Inconel 718 Tensile Performance
  m= 12 kw=  0 | PMC10744685 | Change in Mechanical Properties of Laser Powder Bed Fused AlSi7Mg Alloy during L
  m= 45 kw=  0 | PMC9611566 | Approach to Estimate the Phase Formation and the Mechanical Properties of Alloys
  m= 66 kw=  4 | PMC12692834 | Comparative Study on Intermediate-Temperature Deformation Mechanisms of Inconel 
  m= 31 kw=  3 | PMC10856223 | M

In [25]:
# --- Draft evaluation set (answers NOT yet verified by hand) ---
eval_set_v2 = [
    {"q": "What laser power, scan speed and hatch spacing worked best when a copper alloy containing chromium and zirconium was remelted?",
     "gold_pmcid": "PMC10856295", "answer_keys": ["330", "400", "0.09"]},
    {"q": "What tensile strength did stainless steel samples reach when printed lengthwise with 50 W laser power?",
     "gold_pmcid": "PMC12786895", "answer_keys": ["550"]},
    {"q": "By how much did the electrical conductivity of an as-printed copper coil fall short of standard copper?",
     "gold_pmcid": "PMC10342347", "answer_keys": ["20%"]},
    {"q": "Which techniques were used to examine channels printed without support in a maraging steel?",
     "gold_pmcid": "PMC11901190", "answer_keys": ["computed tomography"]},
    {"q": "At which temperatures were tensile properties measured for the heat-treated nickel-based superalloy?",
     "gold_pmcid": "PMC10934370", "answer_keys": ["900"]},
    {"q": "Which intermetallic phase strengthens the copper-chromium-niobium alloy?",
     "gold_pmcid": "PMC10998225", "answer_keys": ["Cr2Nb"]},
    {"q": "What spatial resolution was achieved when imaging laser melting with an X-ray free-electron laser?",
     "gold_pmcid": "PMC12067346", "answer_keys": ["0.9"]},
    {"q": "What range of wall thicknesses was studied for the titanium alloy fabricated under identical processing conditions?",
     "gold_pmcid": "PMC13257545", "answer_keys": ["30 mm"]},
]

# --- Evaluation functions ---
def hit_chunk(index, item, k):
    """True if a top-k chunk comes from the gold paper AND contains all answer keys."""
    scores = index.get_scores(tokenize(item["q"]))
    for i in scores.argsort()[::-1][:k]:
        c = chunks_v2[i]
        if c["pmcid"] == item["gold_pmcid"] and all(
                key.lower() in c["text"].lower() for key in item["answer_keys"]):
            return True
    return False

def hit_paper(index, item, k):
    """True if any top-k chunk comes from the gold paper."""
    scores = index.get_scores(tokenize(item["q"]))
    return any(chunks_v2[i]["pmcid"] == item["gold_pmcid"]
               for i in scores.argsort()[::-1][:k])

def evaluate(index, eval_list, ks=(1, 3, 5, 10)):
    out = {}
    for k in ks:
        out[f"paper@{k}"] = round(sum(hit_paper(index, it, k) for it in eval_list) / len(eval_list), 2)
        out[f"chunk@{k}"] = round(sum(hit_chunk(index, it, k) for it in eval_list) / len(eval_list), 2)
    return out

# --- Run: skip questions whose gold paper is not in the index ---
in_index = {c["pmcid"] for c in chunks_v2}
missing = [i["gold_pmcid"] for i in eval_set_v2 if i["gold_pmcid"] not in in_index]
print("Gold papers missing from index:", missing)

usable = [i for i in eval_set_v2 if i["gold_pmcid"] in in_index]
print("Usable questions:", len(usable), "of", len(eval_set_v2))

for name, idx in [("v1 text only", bm25_v1), ("v2 title+section+text", bm25_v2)]:
    print(name, evaluate(idx, usable))

Gold papers missing from index: []
Usable questions: 8 of 8
v1 text only {'paper@1': 0.75, 'chunk@1': 0.62, 'paper@3': 0.88, 'chunk@3': 0.75, 'paper@5': 0.88, 'chunk@5': 0.88, 'paper@10': 0.88, 'chunk@10': 0.88}
v2 title+section+text {'paper@1': 0.88, 'chunk@1': 0.75, 'paper@3': 0.88, 'chunk@3': 0.75, 'paper@5': 0.88, 'chunk@5': 0.75, 'paper@10': 0.88, 'chunk@10': 0.75}


In [26]:
import csv, json
from rank_bm25 import BM25Okapi

# Papers manually excluded after reading their titles (outside the LPBF experimental scope)
EXCLUDE = {"PMC13027603",   # laser wire directed energy deposition, not powder bed fusion
           "PMC13467613",   # PLA polymer metamaterials
           "PMC13591421"}   # graphene/aluminium composites review

rag_ids = [p for p in mentions
           if p not in EXCLUDE and (mentions[p] >= 6 or (mentions[p] >= 3 and kw[p] >= 5))]
extract_ids = [p for p in mentions
               if p not in EXCLUDE and mentions[p] >= 3 and kw[p] >= 5]
print("RAG corpus:", len(rag_ids), "| Extraction set:", len(extract_ids))

def save_list(path, ids):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["pmcid", "title", "lpbf_mentions", "kw_hits"])
        for p in ids:
            writer.writerow([p, title_of[p], mentions[p], kw[p]])

save_list("data/rag_papers.csv", rag_ids)
save_list("data/extraction_papers.csv", extract_ids)

# Rebuild the retrieval index on the RAG corpus
all_chunks = []
for p in rag_ids:
    paper = json.load(open(f"data/processed/{p}.json", encoding="utf-8"))
    all_chunks.extend(make_chunks(p, paper))

chunks_v2 = []
for c in all_chunks:
    chunks_v2.extend(split_long(c))
print("Chunks:", len(chunks_v2))

bm25_v1 = BM25Okapi([tokenize(c["text"]) for c in chunks_v2])
bm25_v2 = BM25Okapi([tokenize(with_header(c)) for c in chunks_v2])

in_index = {c["pmcid"] for c in chunks_v2}
print("Gold papers missing from index:",
      [i["gold_pmcid"] for i in eval_set_v2 if i["gold_pmcid"] not in in_index])

for name, idx in [("v1 text only", bm25_v1), ("v2 title+section+text", bm25_v2)]:
    print(name, evaluate(idx, eval_set_v2))

RAG corpus: 307 | Extraction set: 201
Chunks: 13384
Gold papers missing from index: []
v1 text only {'paper@1': 0.75, 'chunk@1': 0.62, 'paper@3': 0.88, 'chunk@3': 0.75, 'paper@5': 0.88, 'chunk@5': 0.88, 'paper@10': 0.88, 'chunk@10': 0.88}
v2 title+section+text {'paper@1': 0.88, 'chunk@1': 0.75, 'paper@3': 0.88, 'chunk@3': 0.75, 'paper@5': 0.88, 'chunk@5': 0.75, 'paper@10': 0.88, 'chunk@10': 0.75}


In [28]:
import numpy as np, torch, os
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

texts = [with_header(c) for c in chunks_v2]
emb = embedder.encode(texts, batch_size=64, normalize_embeddings=True,
                      show_progress_bar=True)
os.makedirs("data/index", exist_ok=True)
np.save("data/index/chunk_embeddings.npy", emb)    # Saved so it is not recomputed
print("Embeddings shape:", emb.shape)

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

class DenseIndex:
    """Wrapper so the dense index can reuse evaluate() written for BM25."""
    def get_scores(self, query_tokens):
        query = " ".join(query_tokens)
        q = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True)
        return emb @ q[0]

dense = DenseIndex()
usable = [i for i in eval_set_v2 if i["gold_pmcid"] in {c["pmcid"] for c in chunks_v2}]
for name, idx in [("BM25 v2", bm25_v2), ("Dense bge-small", dense)]:
    print(name, evaluate(idx, usable))

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/210 [00:00<?, ?it/s]

Embeddings shape: (13384, 384)
BM25 v2 {'paper@1': 0.88, 'chunk@1': 0.75, 'paper@3': 0.88, 'chunk@3': 0.75, 'paper@5': 0.88, 'chunk@5': 0.75, 'paper@10': 0.88, 'chunk@10': 0.75}
Dense bge-small {'paper@1': 1.0, 'chunk@1': 0.38, 'paper@3': 1.0, 'chunk@3': 0.75, 'paper@5': 1.0, 'chunk@5': 0.75, 'paper@10': 1.0, 'chunk@10': 0.88}


In [29]:
for item in usable:
    scores = dense.get_scores(tokenize(item["q"]))
    order = scores.argsort()[::-1]
    top = chunks_v2[order[0]]
    paper_ok = top["pmcid"] == item["gold_pmcid"]
    keys_ok = all(k.lower() in top["text"].lower() for k in item["answer_keys"])
    if paper_ok and not keys_ok:
        print("Q:", item["q"][:90])
        print("   gold:", item["gold_pmcid"], "| keys:", item["answer_keys"])
        print("   top-1 section:", top["section"][:60], "| type:", top["type"])
        print("   top-1 text:", top["text"][:300].replace("\n", " "))
        print()

Q: What laser power, scan speed and hatch spacing worked best when a copper alloy containing 
   gold: PMC10856295 | keys: ['330', '400', '0.09']
   top-1 section: 1. Introduction | type: text
   top-1 text: Copper and copper alloys have excellent mechanical and physical properties, as well as high electrical and thermal conductivities. They are widely utilized in electronics, manufacturing, national security, aerospace, and other industries and significantly influence both the economy and technological

Q: What tensile strength did stainless steel samples reach when printed lengthwise with 50 W 
   gold: PMC12786895 | keys: ['550']
   top-1 section: 3. Results and Discussion > 3.1. Tensile Test | type: text
   top-1 text: Doubling the laser power and scanning speed within the same print configuration changed the mechanical properties (Figure 10). For samples printed in the longitudinal configuration, the yield strength and tensile strength varied within the range of 369–408 MPa and 53

In [31]:
import numpy as np

def bm25_scores(q):
    return bm25_v2.get_scores(tokenize(q))

def dense_scores_raw(q):
    qv = embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True)
    return emb @ qv[0]

def to_ranks(scores):
    """Convert scores to ranks (1 = best) for every chunk."""
    order = np.argsort(-scores)
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks

def rrf_scores(q, k=60):
    """Reciprocal rank fusion of BM25 and dense rankings."""
    return 1.0 / (k + to_ranks(bm25_scores(q))) + 1.0 / (k + to_ranks(dense_scores_raw(q)))

def answer_rank(score_fn, item):
    """Rank of the first chunk from the gold paper containing ALL answer keys."""
    scores = score_fn(item["q"])
    for r, i in enumerate(np.argsort(-scores), 1):
        c = chunks_v2[i]
        if c["pmcid"] == item["gold_pmcid"] and all(
                key.lower() in c["text"].lower() for key in item["answer_keys"]):
            return r
    return None

methods = {"BM25 v2": bm25_scores, "Dense (raw query)": dense_scores_raw, "Hybrid RRF": rrf_scores}
all_ranks = {name: [answer_rank(fn, it) for it in usable] for name, fn in methods.items()}

print("Answer-chunk rank per question:")
for name, ranks in all_ranks.items():
    print(f"  {name:18s}", ranks)

print("\nSummary:")
for name, ranks in all_ranks.items():
    n = len(ranks)
    rec = {f"R@{k}": round(sum(r is not None and r <= k for r in ranks) / n, 2) for k in (1, 3, 5, 10)}
    mrr = round(sum(1 / r for r in ranks if r) / n, 3)
    print(f"  {name:18s}", rec, "| MRR:", mrr)

Answer-chunk rank per question:
  BM25 v2            [1005, 1, 1, 1, 12, 1, 1, 1]
  Dense (raw query)  [13, 2, 1, 1, 1, 1, 2, 19]
  Hybrid RRF         [32, 1, 1, 1, 5, 1, 1, 4]

Summary:
  BM25 v2            {'R@1': 0.75, 'R@3': 0.75, 'R@5': 0.75, 'R@10': 0.75} | MRR: 0.761
  Dense (raw query)  {'R@1': 0.5, 'R@3': 0.75, 'R@5': 0.75, 'R@10': 0.75} | MRR: 0.641
  Hybrid RRF         {'R@1': 0.62, 'R@3': 0.62, 'R@5': 0.88, 'R@10': 0.88} | MRR: 0.685


In [32]:
!pip install -q transformers accelerate

In [33]:
import time, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, torch_dtype=torch.float16, device_map="auto")

def retrieve(question, k=5):
    """Return the top-k chunks using hybrid (BM25 + dense) retrieval."""
    scores = rrf_scores(question)
    return [chunks_v2[i] for i in np.argsort(-scores)[:k]]

def build_messages(question, passages):
    context = "\n\n".join(
        f"[{n}] ({p['title'][:60]} | {p['section'][:40]})\n{p['text']}"
        for n, p in enumerate(passages, 1))
    system = ("You answer questions about laser powder bed fusion using ONLY the "
              "numbered passages provided. Cite passage numbers like [1]. "
              "If the passages do not contain the answer, reply exactly: "
              "'Not found in the provided passages.' Do not use outside knowledge.")
    user = f"Passages:\n{context}\n\nQuestion: {question}"
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]

def answer(question, k=5, max_new_tokens=200):
    passages = retrieve(question, k)
    text = tok.apply_chat_template(build_messages(question, passages),
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    start = time.time()
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return reply, passages, time.time() - start

# Try three questions from the draft evaluation set
for item in usable[:3]:
    reply, passages, secs = answer(item["q"])
    print("Q:", item["q"])
    print("A:", reply)
    print("Sources:", [p["pmcid"] for p in passages], "| gold:", item["gold_pmcid"])
    print(f"Time: {secs:.1f}s\n")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Q: What laser power, scan speed and hatch spacing worked best when a copper alloy containing chromium and zirconium was remelted?
A: Not found in the provided passages.
Sources: ['PMC10616214', 'PMC10856295', 'PMC12654406', 'PMC13514143', 'PMC10056913'] | gold: PMC10856295
Time: 2.2s

Q: What tensile strength did stainless steel samples reach when printed lengthwise with 50 W laser power?
A: 550 MPa [1]
Sources: ['PMC12786895', 'PMC12786895', 'PMC12387571', 'PMC12786895', 'PMC12786895'] | gold: PMC12786895
Time: 1.2s

Q: By how much did the electrical conductivity of an as-printed copper coil fall short of standard copper?
A: 20%
Sources: ['PMC10342347', 'PMC10342347', 'PMC11396442', 'PMC10342347', 'PMC10342347'] | gold: PMC10342347
Time: 0.9s



In [34]:
import re, time, torch, numpy as np

def run_llm(messages, max_new_tokens=200):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    start = time.time()
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return reply.strip(), time.time() - start

def rag_answer(score_fn, question, k=5):
    scores = score_fn(question)
    passages = [chunks_v2[i] for i in np.argsort(-scores)[:k]]
    reply, secs = run_llm(build_messages(question, passages))
    return reply, passages, secs

def closed_book_answer(question):
    msgs = [{"role": "system", "content":
             "Answer the question about laser powder bed fusion briefly. "
             "If you do not know, say 'I do not know.'"},
            {"role": "user", "content": question}]
    reply, secs = run_llm(msgs)
    return reply, [], secs

def is_abstain(reply):
    r = reply.lower()
    return "not found" in r or "do not know" in r

def score_one(reply, passages, item):
    return {
        "correct": all(k.lower() in reply.lower() for k in item["answer_keys"]),
        "abstain": is_abstain(reply),
        "cited": bool(re.search(r"\[\d+\]", reply)),
        # was a passage that contains the answer actually retrieved?
        "retrieved": any(p["pmcid"] == item["gold_pmcid"] and
                         all(k.lower() in p["text"].lower() for k in item["answer_keys"])
                         for p in passages),
    }

configs = {
    "closed-book":  None,
    "BM25 + LLM":   bm25_scores,
    "Dense + LLM":  dense_scores_raw,
    "Hybrid + LLM": rrf_scores,
}

torch.cuda.reset_peak_memory_stats()
results, replies = {}, {}
for name, fn in configs.items():
    rows = []
    for item in usable:
        if fn is None:
            reply, passages, secs = closed_book_answer(item["q"])
        else:
            reply, passages, secs = rag_answer(fn, item["q"])
        s = score_one(reply, passages, item)
        s["secs"] = secs
        rows.append(s)
        replies[(name, item["q"])] = reply
    results[name] = rows

n = len(usable)
print(f"{'config':14s} correct  abstain  cited  retrieved  avg_secs")
for name, rows in results.items():
    print(f"{name:14s} {sum(r['correct'] for r in rows)}/{n}      "
          f"{sum(r['abstain'] for r in rows)}/{n}      "
          f"{sum(r['cited'] for r in rows)}/{n}    "
          f"{sum(r['retrieved'] for r in rows)}/{n}        "
          f"{np.mean([r['secs'] for r in rows]):.1f}")
print("Peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

# Print the closed-book answers to see whether the model makes things up
print("\nClosed-book answers:")
for item in usable:
    print("-", item["q"][:70])
    print("   ->", replies[("closed-book", item["q"])][:150])

config         correct  abstain  cited  retrieved  avg_secs
closed-book    0/8      8/8      0/8    0/8        0.9
BM25 + LLM     5/8      1/8      6/8    6/8        2.9
Dense + LLM    5/8      1/8      4/8    6/8        2.8
Hybrid + LLM   6/8      1/8      5/8    7/8        2.4
Peak GPU memory (GB): 6.9

Closed-book answers:
- What laser power, scan speed and hatch spacing worked best when a copp
   -> I do not know.
- What tensile strength did stainless steel samples reach when printed l
   -> I do not know.
- By how much did the electrical conductivity of an as-printed copper co
   -> I do not know.
- Which techniques were used to examine channels printed without support
   -> I do not know.
- At which temperatures were tensile properties measured for the heat-tr
   -> I do not know.
- Which intermetallic phase strengthens the copper-chromium-niobium allo
   -> I do not know.
- What spatial resolution was achieved when imaging laser melting with a
   -> I do not know.
- What range o

In [35]:
name = "Hybrid + LLM"
for item, r in zip(usable, results[name]):
    if not r["correct"]:
        print("Q:", item["q"])
        print("   reply:", replies[(name, item["q"])][:200])
        print("   answer chunk retrieved:", r["retrieved"], "| abstained:", r["abstain"])

Q: What laser power, scan speed and hatch spacing worked best when a copper alloy containing chromium and zirconium was remelted?
   reply: Not found in the provided passages.
   answer chunk retrieved: False | abstained: True
Q: At which temperatures were tensile properties measured for the heat-treated nickel-based superalloy?
   reply: [3] states that the microstructures and tensile properties of the L-PBF IN718 alloy were investigated at both ambient temperature and elevated temperatures. Specifically, [4] mentions that at 650 °C, 
   answer chunk retrieved: True | abstained: False


In [36]:
def closed_book_forced(question):
    msgs = [{"role": "system", "content":
             "Answer the question about laser powder bed fusion with a specific answer, "
             "even if you are unsure. Never say you do not know."},
            {"role": "user", "content": question}]
    reply, secs = run_llm(msgs)
    return reply, [], secs

forced = []
for item in usable:
    reply, _, _ = closed_book_forced(item["q"])
    ok = all(k.lower() in reply.lower() for k in item["answer_keys"])
    forced.append(ok)
    print("-", item["q"][:60], "| correct:", ok)
    print("   ->", reply[:160])
print("Forced closed-book correct:", sum(forced), "/", len(usable))

- What laser power, scan speed and hatch spacing worked best w | correct: False
   -> Determining the exact laser power, scan speed, and hatch spacing that worked best for remelting a copper alloy containing chromium and zirconium in a laser powd
- What tensile strength did stainless steel samples reach when | correct: False
   -> I'm unable to provide a specific tensile strength value for stainless steel samples printed lengthwise with 50 W laser power using laser powder bed fusion (LPBF
- By how much did the electrical conductivity of an as-printed | correct: False
   -> I do not have specific information regarding the exact electrical conductivity shortfall of an as-printed copper coil compared to standard copper. However, I ca
- Which techniques were used to examine channels printed witho | correct: True
   -> To examine channels printed without support in maraging steel using laser powder bed fusion (LPBF), researchers typically employed a combination of non-destruct
- At which te

In [37]:
for item in usable:
    if any(ch.isdigit() for ch in "".join(item["answer_keys"])):
        reply, _, _ = closed_book_forced(item["q"])
        print("Q:", item["q"][:80])
        print("keys:", item["answer_keys"])
        print("reply:", reply)
        print("-" * 60)

Q: What laser power, scan speed and hatch spacing worked best when a copper alloy c
keys: ['330', '400', '0.09']
reply: Determining the exact laser power, scan speed, and hatch spacing that worked best for remelting a copper alloy containing chromium and zirconium in a laser powder bed fusion process requires specific experimental data which is not provided here. However, generally, these parameters can be optimized through a series of trials to achieve the desired microstructure and mechanical properties.

Typically, higher laser powers can increase melting rates but may also lead to overheating or porosity if not managed properly. A balance between energy density and melt pool size is crucial. The scan speed affects the cooling rate and can influence the grain structure; it should be adjusted to ensure sufficient cooling without causing excessive thermal stresses.

Hatch spacing is important for maintaining uniformity and minimizing defects like undercutting or overlapping layers. It

In [38]:
import json, re

FIELDS = ["material", "laser_power_W", "scan_speed_mm_s", "layer_thickness_um",
          "hatch_spacing_um", "preheat_temperature_C", "relative_density_pct"]

EXTRACTION_SYSTEM = """You extract laser powder bed fusion (LPBF) process parameters
from a materials science paper.

Extract ONLY the parameters used in THIS paper's own printing experiments.
Do NOT extract:
- the maximum power or specification of the machine (for example "a 500 W laser")
- parameters used only in simulations or finite element models
- values taken from other papers (marked by citations such as [13])
- derived quantities such as volumetric energy density (J/mm3)

Return JSON only, with exactly these keys:
material (string or null),
laser_power_W (list of numbers or null),
scan_speed_mm_s (list of numbers or null),
layer_thickness_um (list of numbers or null),
hatch_spacing_um (list of numbers or null),
preheat_temperature_C (list of numbers or null),
relative_density_pct (list of numbers or null).

If the paper does not state a value, use null. Do not guess. Output nothing except the JSON."""

def build_extraction_input(paper, max_method_words=2000, max_table_words=800):
    """Methods-like paragraphs plus the tables those paragraphs refer to."""
    keys = ("method", "experiment", "material", "procedure")
    paras = [p for p in paper["paragraphs"] if any(k in p["section"].lower() for k in keys)]
    if not paras:                                   # fallback: start of the paper
        paras = paper["paragraphs"]
    method_text = " ".join(" ".join(p["text"] for p in paras).split()[:max_method_words])

    cited = set(re.findall(r"table\s*(\d+)", method_text.lower()))
    table_lines, used = [], 0
    for t in paper["tables"]:
        num = re.search(r"\d+", t["label"] or "")
        if num and num.group(0) in cited:
            lines = [(t["label"] + " " + t["caption"]).strip()]
            lines += [" | ".join(r) for r in t["rows"]]
            block = "\n".join(lines)
            used += len(block.split())
            if used > max_table_words:
                break
            table_lines.append(block)
    return method_text + "\n\nTABLES:\n" + "\n\n".join(table_lines)

def parse_json(reply):
    m = re.search(r"\{.*\}", reply, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def extract_params(pmcid, max_new_tokens=300):
    paper = json.load(open(f"data/processed/{pmcid}.json", encoding="utf-8"))
    paper_text = build_extraction_input(paper)
    messages = [{"role": "system", "content": EXTRACTION_SYSTEM},
                {"role": "user", "content": "Paper text:\n" + paper_text}]
    reply, secs = run_llm(messages, max_new_tokens=max_new_tokens)
    n_tokens = len(tok(paper_text)["input_ids"])
    return reply, parse_json(reply), secs, n_tokens

# Trial run on three papers
for pmcid in ["PMC13164529", "PMC12985775", "PMC12985958"]:
    reply, parsed, secs, n_tokens = extract_params(pmcid)
    print(pmcid, "| input tokens:", n_tokens, "| time: %.1fs" % secs, "| valid JSON:", parsed is not None)
    print(reply)
    print("-" * 60)

PMC13164529 | input tokens: 2681 | time: 10.3s | valid JSON: True
```json
{
  "material": "IN738LC",
  "laser_power_W": [175, 200, 225, 250, 275],
  "scan_speed_mm_s": [1000, 900, 800, 900, 1000],
  "layer_thickness_um": [30],
  "hatch_spacing_um": [100],
  "preheat_temperature_C": [293]
}
```
------------------------------------------------------------
PMC12985775 | input tokens: 1445 | time: 5.1s | valid JSON: True
```json
{
  "material": "CuNi2.5SiCr",
  "laser_power_W": [67.2],
  "scan_speed_mm_s": null,
  "layer_thickness_um": [40],
  "hatch_spacing_um": [80],
  "preheat_temperature_C": [100]
}
```
------------------------------------------------------------
PMC12985958 | input tokens: 1751 | time: 8.0s | valid JSON: True
{
  "material": "EN AW 7075",
  "laser_power_W": [150, 200, 300, 400],
  "scan_speed_mm_s": [600, 800, 1000, 1200, 1400],
  "layer_thickness_um": [30],
  "hatch_spacing_um": [60],
  "preheat_temperature_C": [25, 100, 200, 400]
}
----------------------------------

In [ ]:
from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")   # Only needed once per session

!mkdir -p /content/drive/MyDrive/cp5403/data
# -u copies only files that are new or changed, so repeated runs are fast
!cp -ru /content/data/. /content/drive/MyDrive/cp5403/data/
print("Synced to Google Drive.")